# 4단계: 문장 선택 (Colab, GPU)

3단계에서 재정렬한 **top-5 문서**(원본 HoVer와 같음)의 모든 문장을 후보로 두고, (claim, 문장) 쌍마다 "근거 문장일 확률"을 매겨 상위 문장을 고른다.

- 정답 = `supporting_facts`의 (문서 제목, 문장 번호)
- 문장 번호를 맞추려고 0단계와 똑같이 CoreNLP로 문장을 나눈다
- 평가: 고른 top-k 문장 안에 정답 근거 문장이 **모두** 들어왔는가 (+ 문장 단위 recall/precision)
- 천장: 문서 단계 recall@5 (test 0.558). 문서를 놓치면 그 안의 문장도 고를 수 없다

| 셀 | 역할 |
|---|---|
| 1 | 0단계 데이터 + 3단계 재정렬 결과 불러오기 |
| 2 | CoreNLP 서버 시작, 문서 → 문장 함수 |
| 3 | (claim, 문장) 쌍 만들기. 학습 때는 정답 문서를 항상 포함 (3단계와 같은 이유) |
| 4 | 문장 선택 모델 학습 |
| 5 | 문장 점수 매기기, k별 평가 → val에서 k를 고른다 |
| 6 | 결과 저장 (5단계에서 씀) |

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U transformers datasets accelerate stanza

import json, os, sqlite3, unicodedata
from functools import lru_cache
import numpy as np, pandas as pd

ROOT = '/content/drive/MyDrive/fact-verification'
RAW, PROC = f'{ROOT}/data/raw', f'{ROOT}/data/processed'
WIKI = '/content/wiki_wo_links.db'
if not os.path.exists(WIKI):
    !cp "{RAW}/wiki_wo_links.db" {WIKI}
con = sqlite3.connect(WIKI)

nfc = lambda s: unicodedata.normalize('NFC', s)  # 정답 제목은 NFC, 위키 DB/TF-IDF 제목은 NFD라서 비교 전에 통일
TAG = '_mh'  # 베이스라인: '' / 08 multi-hop 문서 검색 결과를 쓰려면 '_mh'
dfs = {}
for n in ['train', 'val', 'test']:
    d = pd.read_json(f'{PROC}/hover2_{n}.jsonl', lines=True)
    r = pd.read_json(f'{PROC}/hover2_{n}_docret{TAG}.jsonl', lines=True)
    d = d.merge(r, on='uid', validate='1:1')
    d['reranked'] = d.reranked.map(lambda ts: [nfc(t) for t in ts])
    d['gold_docs'] = d.supporting_facts.map(lambda sf: list(dict.fromkeys(nfc(t) for t, _ in sf)))
    d['gold_sents'] = d.supporting_facts.map(lambda sf: {(nfc(t), i) for t, i in sf})
    dfs[n] = d
{n: len(d) for n, d in dfs.items()}

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 174.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 134.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 56.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.2/794.2 kB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 51.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


{'train': 8166, 'val': 886, 'test': 1126}

In [2]:
!which java || apt-get -qq install -y openjdk-17-jre-headless
import stanza
from stanza.server import CoreNLPClient, StartServer

os.environ['CORENLP_HOME'] = '/content/corenlp'
if not os.path.exists('/content/corenlp'):
    stanza.install_corenlp(dir='/content/corenlp')
# 9000번 포트는 Colab에서 뜨지 않았다. 문서 수만 개를 처리하므로 로그는 끈다
# TRY_START: 앞서 띄운 서버가 9011에 남아 있으면 새로 띄우지 않고 그 서버에 연결한다
client = CoreNLPClient(annotators=['tokenize', 'ssplit'], timeout=60000, memory='4G',
                       be_quiet=True, endpoint='http://localhost:9011', start_server=StartServer.TRY_START)
client.start()

@lru_cache(maxsize=None)
def get_sents(title):
    row = con.execute('SELECT text FROM documents WHERE id = ?', (unicodedata.normalize('NFD', title),)).fetchone()
    if row is None:
        return ()
    para = row[0]
    ann = client.annotate(para, output_format='json')
    return tuple(para[s['tokens'][0]['characterOffsetBegin']:s['tokens'][-1]['characterOffsetEnd']]
                 for s in ann['sentences'])

get_sents('Peder Severin Krøyer')

/usr/bin/java


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")
INFO:stanza:Installing CoreNLP 4.6.0 into /content/corenlp


stanford-corenlp-latest.zip: reconstructing file:   0%|          |  0.00B /  511MB            

stanford-corenlp-latest.zip: downloading bytes:           |  0.00B            

INFO:stanza:Downloaded file to /content/corenlp/corenlp.zip
INFO:stanza:Installed CoreNLP 4.6.0 into /content/corenlp
INFO:stanza:Writing properties to tmp file: corenlp_server-d4d675cba7064af5.props
INFO:stanza:Starting server with command: java -Xmx4G -cp /content/corenlp/* edu.stanford.nlp.pipeline.StanfordCoreNLPServer -port 9011 -timeout 60000 -threads 5 -maxCharLength 100000 -quiet True -serverProperties corenlp_server-d4d675cba7064af5.props -annotators tokenize,ssplit -preload -outputFormat serialized


("Peder Severin Krøyer ['peːdə 'seveʁin 'kʁojə] (23 July 1851 – 21 November 1909), professionally known as P. S. Krøyer, was a Danish painter.",
 'He is one of the best known and beloved, and the most colorful of the Skagen Painters, a community of Danish and Nordic artists who lived, gathered, or worked in Skagen, Denmark, especially during the final decades of the 19th century.',
 'Krøyer was the unofficial leader of the group.')

In [3]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL = 'roberta-base'
MAX_LEN = 128
K_DOC = 5
tok = AutoTokenizer.from_pretrained(MODEL)

def make_pairs(d, train):
    rows = []
    for r in d.itertuples():
        titles = list(r.reranked[:K_DOC])
        if train:
            titles = (r.gold_docs + [t for t in titles if t not in r.gold_docs])[:K_DOC]
        for t in titles:
            for i, s in enumerate(get_sents(t)):
                rows.append({'uid': r.uid, 'claim': r.claim, 'title': t, 'sid': i, 'sent': s,
                             'labels': int((t, i) in r.gold_sents)})
    return pd.DataFrame(rows)

pairs = {n: make_pairs(d, n == 'train') for n, d in dfs.items()}

def encode(b):
    # 문장만으로는 "He", "It" 같은 대명사가 누구인지 모르니 문서 제목을 앞에 붙인다
    return tok(b['claim'], [f'{t}. {s}' for t, s in zip(b['title'], b['sent'])], truncation=True, max_length=MAX_LEN)  # 긴 쪽부터 자른다 (claim이 길어도 오류 없음)

ds = {n: Dataset.from_pandas(p[['claim', 'title', 'sent', 'labels']]).map(encode, batched=True, remove_columns=['claim', 'title', 'sent'])
      for n, p in pairs.items()}
{n: (len(p), round(float(p.labels.mean()), 3)) for n, p in pairs.items()}  # (쌍 개수, 양성 비율)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/138410 [00:00<?, ? examples/s]

Map:   0%|          | 0/14833 [00:00<?, ? examples/s]

Map:   0%|          | 0/18909 [00:00<?, ? examples/s]

{'train': (138410, 0.143), 'val': (14833, 0.133), 'test': (18909, 0.13)}

In [4]:
import torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
args = TrainingArguments(
    output_dir='/content/out/sent_select',
    num_train_epochs=1, learning_rate=2e-5, per_device_train_batch_size=32, per_device_eval_batch_size=256,
    warmup_steps=0.1, weight_decay=0.01, fp16=True, seed=42,
    save_strategy='no', logging_steps=200, report_to='none',
)
trainer = Trainer(model=model, args=args, train_dataset=ds['train'], data_collator=DataCollatorWithPadding(tok))
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
200,0.450646
400,0.311569
600,0.271935
800,0.248348
1000,0.246837
1200,0.232824
1400,0.231944
1600,0.225582
1800,0.230083
2000,0.206059


TrainOutput(global_step=4326, training_loss=0.22968720574098994, metrics={'train_runtime': 100.0795, 'train_samples_per_second': 1383.001, 'train_steps_per_second': 43.226, 'total_flos': 8259725110941960.0, 'train_loss': 0.22968720574098994, 'epoch': 1.0})

In [5]:
TOP_SAVE = 10  # 5단계에서 k를 바꿔 볼 수 있게 넉넉히 저장

def rank(n):
    p = pairs[n].copy()
    logits = trainer.predict(ds[n]).predictions
    p['score'] = torch.softmax(torch.tensor(logits), -1)[:, 1].numpy()
    p = p.sort_values('score', ascending=False).groupby('uid', sort=False).head(TOP_SAVE)
    ranked = {uid: g[['title', 'sid', 'sent', 'score']].values.tolist() for uid, g in p.groupby('uid', sort=False)}
    return dfs[n].uid.map(lambda u: ranked.get(u, []))

for n in ['train', 'val', 'test']:
    dfs[n]['pred_sents'] = rank(n)

def sent_metrics(d, k):
    allin, rec, prec = [], [], []
    for gold, pred in zip(d.gold_sents, d.pred_sents):
        top = {(t, int(i)) for t, i, *_ in pred[:k]}
        hit = len(gold & top)
        allin.append(gold <= top); rec.append(hit / len(gold)); prec.append(hit / max(len(top), 1))
    return {'all_gold': round(float(np.mean(allin)), 3), 'recall': round(float(np.mean(rec)), 3), 'precision': round(float(np.mean(prec)), 3)}

for n in ['val', 'test']:
    d = dfs[n]
    ceiling = np.mean([set(g) <= set(r[:K_DOC]) for r, g in zip(d.reranked, d.gold_docs)])
    print(n, f'천장(문서 recall@{K_DOC}) {ceiling:.3f}')
    for k in [2, 3, 4, 5]:
        print(f'  top-{k}', sent_metrics(d, k))

val 천장(문서 recall@5) 0.863
  top-2 {'all_gold': 0.42, 'recall': 0.692, 'precision': 0.784}
  top-3 {'all_gold': 0.574, 'recall': 0.786, 'precision': 0.605}
  top-4 {'all_gold': 0.655, 'recall': 0.833, 'precision': 0.487}
  top-5 {'all_gold': 0.714, 'recall': 0.861, 'precision': 0.407}
test 천장(문서 recall@5) 0.836
  top-2 {'all_gold': 0.376, 'recall': 0.659, 'precision': 0.755}
  top-3 {'all_gold': 0.533, 'recall': 0.761, 'precision': 0.594}
  top-4 {'all_gold': 0.641, 'recall': 0.817, 'precision': 0.483}
  top-5 {'all_gold': 0.708, 'recall': 0.852, 'precision': 0.406}


In [6]:
for n, d in dfs.items():
    d[['uid', 'pred_sents']].to_json(f'{PROC}/hover2_{n}_sentret{TAG}.jsonl', orient='records', lines=True, force_ascii=False)
!ls -lh "{PROC}" 

total 59M
-rw------- 1 root root 884K Oct 21 08:33 hover2_test_docret.jsonl
-rw------- 1 root root 547K Oct 21 14:02 hover2_test_docret_mh.jsonl
-rw------- 1 root root 2.4M Oct 21 13:28 hover2_test_e2e_retrieved.jsonl
-rw------- 1 root root 768K Oct 21 06:22 hover2_test.jsonl
-rw------- 1 root root 2.1M Oct 21 12:55 hover2_test_sentret.jsonl
-rw------- 1 root root 2.0M Oct 22 13:14 hover2_test_sentret_mh.jsonl
-rw------- 1 root root 6.3M Oct 21 08:33 hover2_train_docret.jsonl
-rw------- 1 root root 4.2M Oct 21 14:02 hover2_train_docret_mh.jsonl
-rw------- 1 root root 5.5M Oct 21 06:22 hover2_train.jsonl
-rw------- 1 root root  15M Oct 21 12:55 hover2_train_sentret.jsonl
-rw------- 1 root root  15M Oct 22 13:14 hover2_train_sentret_mh.jsonl
-rw------- 1 root root 697K Oct 21 08:33 hover2_val_docret.jsonl
-rw------- 1 root root 430K Oct 21 14:02 hover2_val_docret_mh.jsonl
-rw------- 1 root root 612K Oct 21 06:22 hover2_val.jsonl
-rw------- 1 root root 1.6M Oct 21 12:55 hover2_val_sentret